# A03. AI가입자 재사용률 분석

- 재사용 기준: 기본 2일 이상 사용
- 가입시점이 다른 문제를 누적 재사용률 곡선으로 보정

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')

root = Path.cwd()
refactor_dir = root if root.name == 'refactor' else (root / 'analysis' / 'refactor')
if not refactor_dir.exists():
    refactor_dir = Path('/workspace/analysis/refactor')
if str(refactor_dir) not in sys.path:
    sys.path.insert(0, str(refactor_dir))

import common
print('refactor_dir =', refactor_dir)

In [ ]:
PROFILE_FILE = '/home/jovyan/cjs/refactor2/data/user.csv'
CHAT_FILE = '/home/jovyan/cjs/refactor2/data/chat.csv'
REUSE_MIN_DAYS = 2

profile = common.normalize_profile(common.load_csv(PROFILE_FILE))
chat = common.normalize_chat(common.load_csv(CHAT_FILE))

ai_base = profile[['customer_id', 'ai_signup_date']].dropna(subset=['customer_id', 'ai_signup_date']).drop_duplicates('customer_id')
chat2 = chat.dropna(subset=['customer_id', 'chat_date']).merge(ai_base, on='customer_id', how='inner')
chat2['day_offset'] = (chat2['chat_date'] - chat2['ai_signup_date']).dt.days
chat2 = chat2[chat2['day_offset'] >= 0].copy()

u = chat2.groupby('customer_id', as_index=False).agg(total_requests=('request_count', 'sum'), use_days=('chat_date', 'nunique'))
user = ai_base.merge(u, on='customer_id', how='left').fillna({'total_requests':0, 'use_days':0})
user['is_reuser'] = user['use_days'] >= REUSE_MIN_DAYS

base_n = user['customer_id'].nunique()
summary = pd.DataFrame([
    {'지표':'AI가입자수', '값':base_n},
    {'지표':'가입 후 1회 이상 사용자', '값':int((user['total_requests']>0).sum())},
    {'지표':f'{REUSE_MIN_DAYS}일 이상 사용자(재사용)', '값':int(user['is_reuser'].sum())},
])
summary['비율'] = summary['값'] / max(base_n, 1)
summary

In [ ]:
curve = []
for d in range(1, 31):
    tmp = chat2[chat2['day_offset'] <= d]
    by_user = tmp.groupby('customer_id', as_index=False).agg(use_days=('chat_date','nunique'))
    reuse_users = int((by_user['use_days'] >= REUSE_MIN_DAYS).sum())
    curve.append({'day': d, 'cum_reuse_rate': reuse_users / max(base_n, 1)})
curve = pd.DataFrame(curve)

fig, axes = plt.subplots(1,2, figsize=(14,4))
sns.barplot(data=summary.iloc[1:], x='지표', y='비율', palette='Set2', ax=axes[0])
axes[0].set_ylim(0,1); axes[0].tick_params(axis='x', rotation=20)
axes[0].set_title('재사용 핵심 비율')

sns.lineplot(data=curve, x='day', y='cum_reuse_rate', marker='o', color='#55a868', ax=axes[1])
axes[1].set_ylim(0,1); axes[1].set_title('가입 후 누적 재사용률 (1~30일)')
plt.tight_layout(); plt.show()